# Hausa News Topic Classifier: Run the Live App in Colab

**How to use**
1. (Optional, faster) `Runtime → Change runtime type → T4 GPU`.
2. Run **Cell 1**. If Colab asks you to restart the runtime, do it, then continue from Cell 2.
3. Run **Cell 2** (loads the model).
4. Run **Cell 3** and open the `https://xxxx.gradio.live` link it prints.

**Notes:** keep this tab open while the app is in use. The public link expires after about 72 hours, and a new link is generated each time you run Cell 3.


In [ ]:
!pip -q install gradio transformers sentencepiece protobuf

In [ ]:
import gradio as gr, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Model on the Hugging Face Hub (must be public).
MODEL_ID = "dmahe09/hausa-news-classifier"

# If the model is NOT on the Hub, mount Google Drive and point to the folder instead:
# from google.colab import drive; drive.mount('/content/drive')
# MODEL_ID = "/content/drive/MyDrive/best_model"

MAX_LEN = 512
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device).eval()
print("Model loaded. Labels:", list(model.config.id2label.values()))

In [ ]:
def predict(headline, text):
    enc = tok((headline or "") + ". " + (text or ""),
              truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, -1)[0].cpu()
    return {model.config.id2label[i]: float(p) for i, p in enumerate(probs)}

# Quick sanity check before launching the app
print(predict("Chelsea 2-4 Arsenal: Gunners ta je ta ragargaji Chelsea a Stamford Bridge", ""))

demo = gr.Interface(
    fn=predict,
    inputs=[gr.Textbox(label="Headline (Hausa)"),
            gr.Textbox(label="Article text (Hausa)", lines=8)],
    outputs=gr.Label(num_top_classes=7, label="Predicted topic"),
    title="Hausa News Topic Classifier",
    description="Fine-tuned AfroXLMR on MasakhaNEWS (Hausa). Categories: business, entertainment, health, politics, religion, sports, technology.",
)

# share=True creates the public https://xxxx.gradio.live link
demo.launch(share=True, debug=True)